# VR教室 API 测试笔记本

整合所有测试脚本，方便快速进行测试运行

## 目录
1. 环境配置
2. 用户登录测试
3. 帖子和评论API测试
4. 管理员审核API测试
5. 捐赠与支付流程测试
6. 订单模块测试

---
## 1. 环境配置

In [ ]:
import requests
import json
import time
import uuid
from IPython.display import JSON, display

# 基础URL配置
BASE_URL = "http://localhost:8080/api"
SERVER_URL = "http://10.86.136.242:8082/api"

# 切换服务器: True 使用服务器, False 使用本地
IS_SERVER = False
URL = SERVER_URL if IS_SERVER else BASE_URL

print(f"当前服务器: {URL}")
print(f"环境: {'服务器' if IS_SERVER else '本地'}")

In [ ]:
# 测试结果存储
test_results = []

# 通用测试函数
def test_api(method, endpoint, body=None, require_auth=False, token=None, use_params=False):
    """
    通用API测试函数
    
    Args:
        method: HTTP方法 (GET, POST, PUT, PATCH, DELETE)
        endpoint: API端点
        body: 请求体或参数
        require_auth: 是否需要认证
        token: 认证token
        use_params: POST请求是否使用查询参数
    """
    headers = {"Content-Type": "application/json"}
    if require_auth and token:
        headers["Authorization"] = f"Bearer {token}"
    
    try:
        full_url = f"{URL}{endpoint}"
        
        if method == "GET":
            response = requests.get(full_url, params=body, headers=headers)
        elif method == "POST":
            if use_params:
                response = requests.post(full_url, params=body, headers=headers)
            else:
                response = requests.post(full_url, json=body, headers=headers)
        elif method == "PUT":
            response = requests.put(full_url, json=body, headers=headers)
        elif method == "PATCH":
            response = requests.patch(full_url, json=body, headers=headers)
        elif method == "DELETE":
            response = requests.delete(full_url, headers=headers)
        else:
            print(f"❌ 不支持的方法: {method}")
            return None
        
        result = {
            "endpoint": endpoint,
            "method": method,
            "status_code": response.status_code,
            "response": None
        }
        
        try:
            result["response"] = response.json()
        except:
            result["response"] = response.text
        
        if response.status_code < 400:
            result["status"] = "Success"
            print(f"✅ {method} {endpoint} - {response.status_code}")
        else:
            result["status"] = "Failed"
            print(f"❌ {method} {endpoint} - {response.status_code}")
        
        test_results.append(result)
        return result
        
    except Exception as e:
        result = {
            "endpoint": endpoint,
            "method": method,
            "status": "Error",
            "error": str(e)
        }
        test_results.append(result)
        print(f"❌ {method} {endpoint} - Error: {str(e)}")
        return result

def generate_unique_suffix():
    """生成唯一后缀"""
    return str(uuid.uuid4()).split('-')[0].upper()

def print_test_summary():
    """打印测试结果汇总"""
    print("\n" + "="*60)
    print("测试结果汇总")
    print("="*60)
    
    success_count = sum(1 for r in test_results if r.get("status") == "Success")
    failed_count = sum(1 for r in test_results if r.get("status") == "Failed")
    error_count = sum(1 for r in test_results if r.get("status") == "Error")
    
    print(f"总测试数: {len(test_results)}")
    print(f"成功: {success_count}")
    print(f"失败: {failed_count}")
    print(f"错误: {error_count}")
    
    if failed_count > 0 or error_count > 0:
        print("\n失败的测试:")
        for result in test_results:
            if result.get("status") in ["Failed", "Error"]:
                print(f"  - {result.get('method')} {result.get('endpoint')}: {result.get('status_code', 'N/A')}")

---
## 2. 用户登录测试

In [ ]:
# 用户登录获取token - 使用手机号登录接口（备用测试接口）
def login_user(phone):
    """使用手机号登录接口（备用测试接口，无需微信code）"""
    response = requests.post(f"{URL}/users/login/phone", json={"phone": phone})
    if response.status_code == 200:
        data = response.json().get("data", {})
        token = data.get("token")
        user = data.get("user", {})
        print(f"✅ 登录成功: {user.get('name')} (ID: {user.get('id')})")
        return token, user
    else:
        print(f"❌ 登录失败: {response.status_code} - {response.text}")
        return None, None

# 微信登录 - 需要真实的微信code
def login_wechat(login_code, phone_code, nick_name=None, avatar_url=None):
    """
    微信登录接口（需要真实的微信code）
    
    Args:
        login_code: 微信登录凭证，用于换取openId
        phone_code: 微信手机号获取凭证，用于换取手机号
        nick_name: 用户昵称（可选）
        avatar_url: 用户头像URL（可选）
    """
    body = {
        "loginCode": login_code,
        "phoneCode": phone_code
    }
    if nick_name:
        body["nickName"] = nick_name
    if avatar_url:
        body["avatarUrl"] = avatar_url
    
    response = requests.post(f"{URL}/users/login", json=body)
    if response.status_code == 200:
        data = response.json().get("data", {})
        token = data.get("token")
        user = data.get("user", {})
        print(f"✅ 微信登录成功: {user.get('name')} (ID: {user.get('id')})")
        return token, user
    else:
        print(f"❌ 微信登录失败: {response.status_code} - {response.text}")
        return None, None

# 测试用户列表
test_users = [
    {"phone": "13800138001", "name": "张三"},
    {"phone": "13800138002", "name": "李四"},
    {"phone": "13800138003", "name": "王五"},
]

# 存储token
tokens = {}

print("=== 测试用户登录 ===")
for user in test_users:
    token, user_info = login_user(user["phone"])
    if token:
        tokens[user["phone"]] = {"token": token, "user": user_info}
    print("-" * 40)

# 设置默认token
default_token = tokens.get("13800138001", {}).get("token")
print(f"\n默认Token已设置: {default_token[:50] if default_token else 'None'}...")

---
## 3. 帖子和评论API测试

In [ ]:
print("=== 测试帖子API ===")

# 确保已登录
if not default_token:
    print("\n--- 首先登录获取token ---")
    token, user = login_user("13800138001")
    if token:
        tokens["13800138001"] = {"token": token, "user": user}
        default_token = token

# 获取帖子列表
print("\n--- 获取帖子列表 ---")
posts_result = test_api("GET", "/posts", {"page": 0})
if posts_result and posts_result.get("status") == "Success":
    display(JSON(posts_result["response"]))

# 获取单个帖子
print("\n--- 获取单个帖子 ---")
test_api("GET", "/posts/1")

# 创建帖子 (需要认证)
print("\n--- 创建帖子 ---")
post_body = {
    "title": f"测试帖子_{generate_unique_suffix()}",
    "content": "这是一个测试帖子的内容",
    "summary": "测试帖子摘要",
    "images": ["https://example.com/test.jpg"],
    "categoryId": 1
}
if default_token:
    test_api("POST", "/posts", post_body, require_auth=True, token=default_token)
else:
    print("❌ 未登录，跳过创建帖子")

In [ ]:
print("=== 测试评论API ===")

# 获取评论列表
print("\n--- 获取评论列表 ---")
test_api("GET", "/comments", {"postId": 1, "page": 0})

# 创建评论 (需要认证)
print("\n--- 创建评论 ---")
comment_body = {
    "content": f"测试评论_{generate_unique_suffix()}",
    "postId": 1
}
if default_token:
    test_api("POST", "/comments", comment_body, require_auth=True, token=default_token)
else:
    print("❌ 未登录，跳过创建评论")

# 获取用户帖子
print("\n--- 获取用户帖子 ---")
if default_token:
    test_api("GET", "/users/posts", {"page": 0}, require_auth=True, token=default_token)
else:
    print("❌ 未登录，跳过获取用户帖子")

# 获取用户评论
print("\n--- 获取用户评论 ---")
if default_token:
    test_api("GET", "/users/comments", {"page": 0}, require_auth=True, token=default_token)
else:
    print("❌ 未登录，跳过获取用户评论")

---
## 4. 管理员审核API测试

In [ ]:
print("=== 测试管理员审核API ===")

# 获取待审核帖子列表
print("\n--- 获取帖子列表(管理员) ---")
admin_posts = test_api("GET", "/admin/posts", {"page": 0, "status": 0})

# 获取待审核评论列表
print("\n--- 获取评论列表(管理员) ---")
admin_comments = test_api("GET", "/admin/comments", {"page": 0, "status": 0})

# 审核帖子
print("\n--- 审核帖子 ---")
test_api("PATCH", "/admin/posts/1", {"status": 1, "rejectReason": None})

# 审核评论
print("\n--- 审核评论 ---")
test_api("PATCH", "/admin/comments/1", {"status": 1, "rejectReason": None})

---
## 5. 捐赠与支付流程测试

In [ ]:
print("=== 测试捐赠与支付流程 ===")

# 确保已登录
if not default_token:
    print("\n--- 首先登录获取token ---")
    token, user = login_user("13800138001")
    if token:
        tokens["13800138001"] = {"token": token, "user": user}
        default_token = token

# 1. 创建捐赠订单
print("\n--- 1. 创建捐赠订单 ---")
donation_body = {
    "seatId": 1,
    "tierId": 1,
    "message": "测试捐赠消息",
    "paymentMethod": "WECHAT"
}
donation_result = None
if default_token:
    donation_result = test_api("POST", "/donation/create", donation_body, require_auth=True, token=default_token)
else:
    print("❌ 未登录，跳过捐赠测试")

if donation_result and donation_result.get("status") == "Success":
    donation_id = donation_result.get("response", {}).get("data", {}).get("id")
    payment_order_no = donation_result.get("response", {}).get("data", {}).get("orderNo")
    print(f"捐赠订单ID: {donation_id}")
    print(f"支付订单号: {payment_order_no}")
else:
    donation_id = None
    payment_order_no = None

In [ ]:
# 2. 获取支付订单信息
print("\n--- 2. 获取支付订单信息 ---")
if payment_order_no:
    test_api("GET", f"/payment/orders/{payment_order_no}")

# 3. 模拟支付回调
print("\n--- 3. 模拟支付回调 ---")
if payment_order_no:
    callback_params = {
        "orderNo": payment_order_no,
        "transactionId": f"TXN{generate_unique_suffix()}",
        "status": 1,
        "sign": "test_sign"
    }
    test_api("POST", "/payment/callback", callback_params, use_params=True)

# 4. 验证支付状态
print("\n--- 4. 验证支付状态 ---")
if payment_order_no:
    test_api("GET", f"/payment/orders/{payment_order_no}")

# 5. 获取用户支付订单列表
print("\n--- 5. 获取用户支付订单列表 ---")
if default_token:
    test_api("GET", "/payment/orders", require_auth=True, token=default_token)
else:
    print("❌ 未登录，跳过获取支付订单列表")

---
## 6. 订单模块测试

In [ ]:
print("=== 测试订单模块 ===")

# 获取教室座位图
def get_room_seats(room_id="1"):
    print(f"\n--- 获取教室座位图 (roomId={room_id}) ---")
    result = test_api("GET", f"/rooms/{room_id}/seats")
    if result and result.get("status") == "Success":
        data = result.get("response", {}).get("data", {})
        seats = data.get("seats", [])
        print(f"教室: {data.get('totalRows')}行 x {data.get('totalCols')}列")
        print(f"座位总数: {len(seats)}")
        
        # 找出可购买的座位 (status=1)
        available_seats = [s for s in seats if s.get("status") == 1]
        print(f"可购买座位: {len(available_seats)}")
        return available_seats
    return []

# 创建订单
def create_order(token, seat_list):
    print(f"\n--- 创建订单 ---")
    if not seat_list:
        print("❌ 没有可购买的座位")
        return None
    
    # 选择前2个座位
    selected_seats = seat_list[:2] if len(seat_list) >= 2 else seat_list[:1]
    
    order_body = {
        "seatList": [
            {"id": str(seat.get("id")), "version": seat.get("version")}
            for seat in selected_seats
        ]
    }
    print(f"选择座位: {[s.get('id') for s in selected_seats]}")
    
    result = test_api("POST", "/orders", order_body, require_auth=True, token=token)
    if result and result.get("status") == "Success":
        return result.get("response", {}).get("data", {}).get("id")
    return None

# 取消订单
def cancel_order(token, order_id):
    print(f"\n--- 取消订单 (orderId={order_id}) ---")
    return test_api("PATCH", f"/orders/{order_id}", {"status": "CANCELLED"}, require_auth=True, token=token)

# 模拟支付回调
def mock_pay_notify(order_id):
    print(f"\n--- 模拟支付回调 (orderId={order_id}) ---")
    return test_api("POST", "/mock/pay/notify", {"orderId": order_id})

In [ ]:
print("\n" + "="*60)
print("完整订单流程测试")
print("="*60)

# 确保已登录
if not default_token:
    print("\n--- 首先登录获取token ---")
    token, user = login_user("13800138001")
    if token:
        tokens["13800138001"] = {"token": token, "user": user}
        default_token = token
    else:
        print("❌ 登录失败，无法继续测试")
        available_seats = []

# 获取座位
if default_token:
    available_seats = get_room_seats("1")
else:
    available_seats = []

if available_seats:
    # 创建订单
    selected_seats = available_seats[:2] if len(available_seats) >= 2 else available_seats[:1]
    
    order_body = {
        "seatList": [
            {"id": str(seat.get("id")), "version": seat.get("version")}
            for seat in selected_seats
        ]
    }
    print(f"\n选择座位: {[s.get('id') for s in selected_seats]}")
    
    order_result = test_api("POST", "/orders", order_body, require_auth=True, token=default_token)
    
    if order_result and order_result.get("status") == "Success":
        order_id = order_result.get("response", {}).get("data", {}).get("id")
        print(f"订单ID: {order_id}")
        
        # 模拟支付
        time.sleep(1)
        test_api("POST", "/mock/pay/notify", {"orderId": order_id})
        
        # 验证座位状态
        time.sleep(1)
        get_room_seats("1")

In [ ]:
print("\n" + "="*60)
print("取消订单流程测试")
print("="*60)

# 确保已登录
token2 = tokens.get("13800138002", {}).get("token")
if not token2:
    print("\n--- 首先登录获取token ---")
    token, user = login_user("13800138002")
    if token:
        tokens["13800138002"] = {"token": token, "user": user}
        token2 = token
    else:
        print("❌ 登录失败，无法继续测试")
        token2 = None

if token2:
    # 获取座位
    available_seats = get_room_seats("1")
else:
    available_seats = []
    
if available_seats:
    selected_seats = available_seats[:1]
    
    order_body = {
        "seatList": [
            {"id": str(seat.get("id")), "version": seat.get("version")}
            for seat in selected_seats
        ]
    }
    print(f"\n选择座位: {[s.get('id') for s in selected_seats]}")
    
    order_result = test_api("POST", "/orders", order_body, require_auth=True, token=token2)
    
    if order_result and order_result.get("status") == "Success":
        order_id = order_result.get("response", {}).get("data", {}).get("id")
        print(f"订单ID: {order_id}")
        
        # 取消订单
        time.sleep(1)
        test_api("PATCH", f"/orders/{order_id}", {"status": "CANCELLED"}, require_auth=True, token=token2)
        
        # 验证座位状态
        time.sleep(1)
        get_room_seats("1")

In [ ]:
print("\n" + "="*60)
print("并发场景测试")
print("="*60)

# 确保已登录
token1 = tokens.get("13800138001", {}).get("token")
token2 = tokens.get("13800138002", {}).get("token")

if not token1:
    print("\n--- 登录用户1 ---")
    token, user = login_user("13800138001")
    if token:
        tokens["13800138001"] = {"token": token, "user": user}
        token1 = token

if not token2:
    print("\n--- 登录用户2 ---")
    token, user = login_user("13800138002")
    if token:
        tokens["13800138002"] = {"token": token, "user": user}
        token2 = token

if token1 and token2:
    # 获取可用座位
    available_seats = get_room_seats("1")
else:
    available_seats = []
    
if available_seats:
    same_seat = available_seats[0]
    print(f"\n两个用户同时选择座位: {same_seat.get('id')}")
    
    order_body = {
        "seatList": [{"id": str(same_seat.get("id")), "version": same_seat.get("version")}]
    }
    
    # 用户1创建订单
    print("\n--- 用户1创建订单 ---")
    result1 = test_api("POST", "/orders", order_body, require_auth=True, token=token1)
    
    # 用户2尝试创建相同座位的订单
    print("\n--- 用户2尝试创建相同座位的订单 ---")
    time.sleep(0.5)
    result2 = test_api("POST", "/orders", order_body, require_auth=True, token=token2)
    
    print("\n并发测试结果:")
    print(f"用户1: {result1.get('status') if result1 else 'None'}")
    print(f"用户2: {result2.get('status') if result2 else 'None'}")

---
## 测试结果汇总

In [ ]:
print_test_summary()

In [ ]:
# 显示详细测试结果
print("\n详细测试结果:")
for i, result in enumerate(test_results, 1):
    print(f"\n{i}. {result.get('method')} {result.get('endpoint')}")
    print(f"   状态: {result.get('status')}")
    print(f"   状态码: {result.get('status_code', 'N/A')}")
    if result.get('error'):
        print(f"   错误: {result.get('error')}")